# 01 — TiNA Data Understanding

**Goal:** verify the source schema, time coverage, missingness, and maintenance/alarm distributions before preprocessing.

The official source is [ari-dasci/OD-TINA](https://github.com/ari-dasci/OD-TINA). The full dataset is too large for routine laptop analysis, so this notebook uses the documented three-day Week 2 pilot. Do not commit source data or oversized notebook outputs.

## Reproducibility record

Verified Week 2 pilot record:

- Source: `ari-dasci/OD-TINA` commit `927b1584ea4aeac2ac7daa084a82f899b114419b`; downloaded 2026-09-21.
- Local file: `data/raw/tina_sample_2017-09-08_to_10.parquet`; SHA-256 `19365a40e99083c37c9909f25b281caad6e47430c52b255c4c24b0db874db947`.
- Selected period: 2017-09-08 00:00:00 through 2017-09-10 23:59:59 UTC. This contiguous three-day window fits in laptop memory and contains maintenance and alarm activity.
- Analyst: PROJ 613 project team (Ganesh Mishra, Lakshya Sharma, and Sagar Patel).

In [1]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import polars as pl
import seaborn as sns

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.build_week2_subset import (
    OUTPUT_PATH as DATA_PATH,
    UPSTREAM_COMMIT,
    main as build_week2_subset,
    sha256,
)
from src.data_utils import scan_tina, validate_event_columns

EXPECTED_SHA256 = '19365a40e99083c37c9909f25b281caad6e47430c52b255c4c24b0db874db947'
if not DATA_PATH.exists() or sha256(DATA_PATH) != EXPECTED_SHA256:
    build_week2_subset()
if sha256(DATA_PATH) != EXPECTED_SHA256:
    raise ValueError('The generated subset does not match the verified official TiNA checksum.')

EXPECTED_PILOT_ROWS = 259_200
print(f'Official TiNA source commit: {UPSTREAM_COMMIT}')
print(f'Verified local subset: {DATA_PATH.relative_to(PROJECT_ROOT)}')
print(f'Expected pilot rows: {EXPECTED_PILOT_ROWS:,}')

Official TiNA source commit: 927b1584ea4aeac2ac7daa084a82f899b114419b
Verified local subset: data/raw/tina_sample_2017-09-08_to_10.parquet
Expected pilot rows: 259,200


## Load lazily and validate

The complete contiguous three-day pilot is loaded below. Findings must still be checked on a broader chronological period before final model claims.

In [2]:
lazy = scan_tina(DATA_PATH)
validate_event_columns(lazy)
schema = lazy.collect_schema()
print(f'Columns: {len(schema.names())}')
schema

Columns: 109


Schema([('FEATURE0', Float64),
        ('FEATURE1', Float64),
        ('FEATURE2', Float64),
        ('FEATURE3', Float64),
        ('FEATURE4', Float64),
        ('FEATURE5', Float64),
        ('FEATURE6', Float64),
        ('FEATURE7', Float64),
        ('FEATURE8', Float64),
        ('FEATURE9', Float64),
        ('FEATURE10', Float64),
        ('FEATURE11', Float64),
        ('FEATURE12', Float64),
        ('FEATURE13', Float64),
        ('FEATURE14', Float64),
        ('FEATURE15', Float64),
        ('FEATURE16', Float64),
        ('FEATURE17', Float64),
        ('FEATURE18', Float64),
        ('FEATURE19', Float64),
        ('FEATURE20', Float64),
        ('FEATURE21', Float64),
        ('FEATURE22', Float64),
        ('FEATURE23', Float64),
        ('FEATURE24', Float64),
        ('FEATURE25', Float64),
        ('FEATURE26', Float64),
        ('FEATURE27', Float64),
        ('FEATURE28', Float64),
        ('FEATURE29', Float64),
        ('FEATURE30', Float64),
        ('FEATURE3

In [3]:
pilot = lazy.collect()
if pilot.height != EXPECTED_PILOT_ROWS:
    raise ValueError(f'Expected {EXPECTED_PILOT_ROWS:,} pilot rows, found {pilot.height:,}.')
print(f'Pilot shape: {pilot.shape}')
pilot.head()

Pilot shape: (259200, 109)


FEATURE0,FEATURE1,FEATURE2,FEATURE3,FEATURE4,FEATURE5,FEATURE6,FEATURE7,FEATURE8,FEATURE9,FEATURE10,FEATURE11,FEATURE12,FEATURE13,FEATURE14,FEATURE15,FEATURE16,FEATURE17,FEATURE18,FEATURE19,FEATURE20,FEATURE21,FEATURE22,FEATURE23,FEATURE24,FEATURE25,FEATURE26,FEATURE27,FEATURE28,FEATURE29,FEATURE30,FEATURE31,FEATURE32,FEATURE33,FEATURE34,FEATURE35,FEATURE36,…,FEATURE72,FEATURE73,FEATURE74,FEATURE75,FEATURE76,FEATURE77,FEATURE78,FEATURE79,FEATURE80,FEATURE81,FEATURE82,FEATURE83,FEATURE84,FEATURE85,FEATURE86,FEATURE87,FEATURE88,FEATURE89,FEATURE90,FEATURE91,FEATURE92,FEATURE93,FEATURE94,FEATURE95,FEATURE96,FEATURE97,FEATURE98,FEATURE99,FEATURE100,FEATURE101,FEATURE102,FEATURE103,FEATURE104,m_id,m_subid,alarms,unix
f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,str,str,i64
0.34802,0.149283,-0.173751,-0.006519,0.739425,0.144183,0.085421,-0.036035,-0.065001,-0.615463,0.085374,0.611267,0.236076,-0.414225,-0.97203,0.739426,0.680813,0.774511,0.058305,-0.490904,0.010176,0.350067,0.3633,-0.467113,0.11935,-0.973038,-1.366509,-1.424929,-1.375139,-1.417426,-1.205329,0.429645,-1.392387,-1.429305,-1.347578,-1.334512,-1.376077,…,2.36555,1.063918,1.055477,-1.597979,"""MI2""",-0.045164,0.373507,2.047508,1.157481,0.308048,1.994843,0.902188,0.083852,0.390231,1.364555,"""LS37""",-1.649163,-1.606435,2.489893,0.914034,0.635845,1.802994,-1.65369,3.514155,-1.268983,1.683808,1.658084,1.134894,0.789802,-0.945786,-1.281975,0.707066,-1.643781,"""none""","""none""","""none""",1504828800
0.34802,0.149283,-0.173751,-0.006519,0.739425,0.144183,0.085421,-0.036035,-0.065001,-0.615463,0.085374,0.611267,0.236076,-0.414225,-0.97203,0.739426,0.680813,0.774511,0.058305,-0.490904,0.010176,0.350067,0.3633,-0.467113,0.11935,-0.973038,-1.366509,-1.424929,-1.375139,-1.417426,-1.205329,0.429645,-1.392387,-1.429305,-1.347578,-1.334512,-1.376077,…,2.36555,1.063918,1.055477,-1.597979,"""MI2""",-0.045164,0.373507,2.047508,1.157481,0.308048,1.994843,0.902188,0.083852,0.390231,1.364555,"""LS37""",-1.649163,-1.606435,2.489893,0.914034,0.635845,1.802994,-1.65369,3.514155,-1.268983,1.683808,1.658084,1.134894,0.789802,-0.945786,-1.281975,0.707066,-1.643781,"""none""","""none""","""none""",1504828801
0.34802,0.149283,-0.173751,-0.006519,0.739425,0.144183,0.085421,-0.036035,-0.065001,-0.615463,0.085374,0.611267,0.236076,-0.414225,-0.97203,0.739426,0.680813,0.774511,0.058305,-0.490904,0.010176,0.350067,0.3633,-0.467113,0.11935,-0.973038,-1.366509,-1.424929,-1.375139,-1.417426,-1.205329,0.429645,-1.392387,-1.429305,-1.347578,-1.334512,-1.376077,…,2.36555,0.961385,0.953469,-1.597979,"""MI2""",-0.045164,0.373507,2.047508,1.157481,0.308048,1.994843,0.902188,0.083852,0.390231,1.364555,"""LS37""",-1.649163,-1.606435,2.489893,0.914034,0.635845,1.802994,-1.65369,3.514155,-1.268983,1.683808,1.658084,1.134894,0.789802,-0.945786,-1.281975,0.707066,-1.643781,"""none""","""none""","""none""",1504828802
0.34802,0.149283,-0.173751,-0.006519,0.739425,0.144183,0.085421,-0.036035,-0.065001,-0.615463,0.085374,0.611267,0.236076,-0.414225,-0.97203,0.739426,0.680813,0.774511,0.058305,-0.490904,0.010176,0.350067,0.3633,-0.467113,0.11935,-0.973038,-1.366509,-1.424929,-1.375139,-1.417426,-1.205329,0.429645,-1.392387,-1.429305,-1.347578,-1.334512,-1.376077,…,2.36555,0.961385,0.953469,-1.597979,"""MI2""",-0.045164,0.373507,2.047508,1.157481,0.308048,1.994843,0.902188,0.083852,0.390231,1.364555,"""LS37""",-1.649163,-1.606435,2.489893,0.914034,0.635845,1.802994,-1.65369,3.514155,-1.268983,1.683808,1.658084,1.134894,0.789802,-0.945786,-1.281975,0.707066,-1.643781,"""none""","""none""","""none""",1504828803
0.34802,0.149283,-0.173751,-0.006519,0.739425,0.144183,0.085421,-0.036035,-0.065001,-0.615463,0.085374,0.611267,0.236076,-0.414225,-0.97203,0.739426,0.680813,0.774511,0.058305,

## Time field and ordering

Confirm the real timestamp column manually if it is not detected. Never build lags, rolling features, or train/test splits until ascending time order is verified.

In [4]:
TIME_CANDIDATES = ('unix', 'timestamp', 'datetime', 'date', 'time', 'Date', 'Time')
time_col = next((name for name in TIME_CANDIDATES if name in pilot.columns), None)
print(f'Detected time column: {time_col!r}')
if time_col:
    print(pilot.select(pl.col(time_col).min().alias('min_time'), pl.col(time_col).max().alias('max_time')))
    print(f'Ascending in pilot: {pilot[time_col].is_sorted()}')
else:
    print('Set time_col to the verified timestamp field before continuing to time-based analysis.')

Detected time column: 'unix'
shape: (1, 2)
┌────────────┬────────────┐
│ min_time   ┆ max_time   │
│ ---        ┆ ---        │
│ i64        ┆ i64        │
╞════════════╪════════════╡
│ 1504828800 ┆ 1505087999 │
└────────────┴────────────┘
Ascending in pilot: True


## Missingness and constant fields

In [5]:
quality = pl.DataFrame({
    'column': pilot.columns,
    'dtype': [str(dtype) for dtype in pilot.dtypes],
    'null_count': [pilot[column].null_count() for column in pilot.columns],
    'n_unique': [pilot[column].n_unique() for column in pilot.columns],
}).with_columns((pl.col('null_count') / pilot.height * 100).alias('null_pct'))
quality.sort('null_pct', descending=True).head(25)

column,dtype,null_count,n_unique,null_pct
str,str,i64,i64,f64
"""FEATURE0""","""Float64""",0,34,0.0
"""FEATURE1""","""Float64""",0,24,0.0
"""FEATURE2""","""Float64""",0,21,0.0
"""FEATURE3""","""Float64""",0,21,0.0
"""FEATURE4""","""Float64""",0,2,0.0
…,…,…,…,…
"""FEATURE20""","""Float64""",0,1,0.0
"""FEATURE21""","""Float64""",0,13,0.0
"""FEATURE22""","""Float64""",0,17,0.0


## Maintenance and alarm distributions

In [6]:
for column in ('m_id', 'm_subid', 'alarms'):
    display(
        pilot.group_by(column)
        .len(name='rows')
        .with_columns((pl.col('rows') / pilot.height * 100).alias('pct'))
        .sort('rows', descending=True)
        .head(20)
    )

m_id,rows,pct
str,u32,f64
"""none""",254206,98.073302
"""M9""",3738,1.44213
"""M4""",1256,0.484568


m_subid,rows,pct
str,u32,f64
"""none""",254206,98.073302
"""MS6""",3738,1.44213
"""MS3""",788,0.304012
"""MS5""",468,0.180556


alarms,rows,pct
str,u32,f64
"""none""",232481,89.691744
"""A111""",5535,2.135417
"""A115""",3684,1.421296
"""A12""",3613,1.393904
"""A97""",2240,0.864198
…,…,…
"""A118""",1801,0.69483
"""A5""",794,0.306327
"""A95""",6,0.002315


## Week 3 data-quality log

These checks use every row in the documented three-day pilot, not the earlier one-day starter sample. The source data stays local.

In [7]:
features = [c for c in pilot.columns if c.startswith('FEATURE')]
floats = [c for c in features if pilot.schema[c] in (pl.Float32, pl.Float64)]
timestamps = pilot['unix']
quality_checks = {
    'rows': pilot.height,
    'columns': pilot.width,
    'unique_timestamps': timestamps.n_unique(),
    'sorted_timestamps': timestamps.is_sorted(),
    'one_second_cadence': (timestamps.diff().drop_nulls() == 1).all(),
    'null_values': sum(pilot.null_count().row(0)),
    'nan_values': sum(pilot.select(pl.col(c).is_nan().sum() for c in floats).row(0)),
    'infinite_values': sum(pilot.select(pl.col(c).is_infinite().sum() for c in floats).row(0)),
    'maintenance_subtype_mismatches': ((pilot['m_id'] == 'none') != (pilot['m_subid'] == 'none')).sum(),
    'constant_features': [c for c in features if pilot[c].n_unique() == 1],
    'string_features': [c for c in features if pilot.schema[c] == pl.String],
}
display(pl.DataFrame({
    'check': ['rows', 'columns', 'unique timestamps', 'sorted timestamps', 'one-second cadence',
              'null values', 'NaN values', 'infinite values', 'maintenance/subtype mismatches'],
    'result': [str(quality_checks[key]) for key in (
        'rows', 'columns', 'unique_timestamps', 'sorted_timestamps', 'one_second_cadence',
        'null_values', 'nan_values', 'infinite_values', 'maintenance_subtype_mismatches'
    )],
}))
print('Constant features in this pilot:', ', '.join(quality_checks['constant_features']))
print('String features:', ', '.join(quality_checks['string_features']))

check,result
str,str
"""rows""","""259200"""
"""columns""","""109"""
"""unique timestamps""","""259200"""
"""sorted timestamps""","""True"""
"""one-second cadence""","""True"""
"""null values""","""0"""
"""NaN values""","""0"""
"""infinite values""","""0"""
"""maintenance/subtype mismatches""","""0"""


Constant features in this pilot: FEATURE9, FEATURE10, FEATURE19, FEATURE20, FEATURE41, FEATURE53, FEATURE62, FEATURE63
String features: FEATURE76, FEATURE87


## Maintenance episodes and recovery gaps

An episode is a maximal run of consecutive seconds with `m_id != 'none'`. Durations and between-episode gaps are descriptive; the 30-minute post-maintenance recovery exclusion remains provisional. Alarm counts are rows, not distinct alarm events.

In [8]:
# Each new active run starts one maintenance episode. This pilot has one-second cadence.
active = pilot['m_id'] != 'none'
starts = active & ~active.shift(1).fill_null(False)
episode_ids = starts.cast(pl.UInt32).cum_sum()
episodes = (pilot.with_columns(episode_id=episode_ids)
    .filter(pl.col('m_id') != 'none')
    .group_by('episode_id').agg(
        pl.col('unix').min().alias('start_unix'),
        pl.col('unix').max().alias('end_unix'),
        pl.col('m_id').unique().sort().alias('maintenance_codes'),
        pl.col('m_subid').unique().sort().alias('subtype_codes'),
        (pl.col('alarms') != 'none').sum().alias('alarm_rows_during'),
    ).sort('start_unix').with_columns(
        (pl.col('end_unix') - pl.col('start_unix') + 1).alias('duration_seconds'),
        (pl.col('start_unix') - pl.col('end_unix').shift(1) - 1).alias('gap_since_previous_seconds'),
    ))
alarm_before = [pilot.filter((pl.col('unix') >= s - 1800) &
                             (pl.col('unix') < s) &
                             (pl.col('alarms') != 'none')).height
                for s in episodes['start_unix'].to_list()]
episodes = episodes.with_columns(pl.Series('alarm_rows_preceding_30m', alarm_before))
event_table = episodes.with_columns(
    pl.from_epoch('start_unix', time_unit='s').dt.strftime('%Y-%m-%d %H:%M:%S').alias('start_utc'),
    pl.from_epoch('end_unix', time_unit='s').dt.strftime('%Y-%m-%d %H:%M:%S').alias('end_utc'),
    pl.col('maintenance_codes').list.join(',').alias('m_id'),
    pl.col('subtype_codes').list.join(',').alias('m_subid'),
).select(
    'episode_id', 'start_utc', 'end_utc', 'duration_seconds',
    'gap_since_previous_seconds', 'm_id', 'm_subid',
    'alarm_rows_preceding_30m', 'alarm_rows_during'
).to_pandas()
display(event_table)
recovery = pl.Series('recovery', [False] * pilot.height)
for end in episodes['end_unix'].to_list():
    recovery = recovery | ((timestamps > end) & (timestamps <= end + 1800) & ~active)
train_end = timestamps[int(pilot.height * 0.70)]
validation_end = timestamps[int(pilot.height * 0.85)]
onsets = episodes['start_unix']
split_counts = {
    'train': (onsets < train_end).sum(),
    'validation': ((onsets >= train_end) & (onsets < validation_end)).sum(),
    'test': (onsets >= validation_end).sum(),
}
print(f'Distinct episodes: {episodes.height}')
print(f'Rows in provisional 30-minute recovery windows: {recovery.sum():,}')
print(f'Episodes in chronological train/validation/test: {split_counts}')

,episode_id,start_utc,end_utc,duration_seconds,gap_since_previous_seconds,m_id,m_subid,alarm_rows_preceding_30m,alarm_rows_during
0,1,2017-09-09 00:33:36,2017-09-09 00:46:09,754,NaN,M9,MS6,0,0
1,2,2017-09-09 07:47:17,2017-09-09 07:53:27,371,25267.0,M9,MS6,0,0
2,3,2017-09-09 16:20:19,2017-09-09 16:28:06,468,30411.0,M4,MS5,0,0
3,4,2017-09-09 20:04:03,2017-09-09 20:17:10,788,12956.0,M4,MS3,0,0
4,5,2017-09-10 03:57:00,2017-09-10 04:17:21,1222,27589.0,M9,MS6,0,0
5,6,2017-09-10 05:20:07,2017-09-10 05:43:17,1391,3765.0,M9,MS6,1355,1391


Distinct episodes: 6
Rows in provisional 30-minute recovery windows: 10,800
Episodes in chronological train/validation/test: {'train': 4, 'validation': 2, 'test': 0}


## Week 3 interpretation and next steps

Verified findings and provisional decisions:

- Verified timestamp and sampling interval: `unix` in UTC seconds; 259,200 unique, ascending timestamps at an exact one-second cadence.
- No null, NaN, infinite, or maintenance/subtype mismatch values were found. Eight constant sensor columns require reassessment on broader data; two string sensor columns require encoding.
- Maintenance/alarm codes present: `m_id` M4/M9; `m_subid` MS3/MS5/MS6; alarms A5/A12/A13/A23/A34/A35/A95/A96/A97/A111/A115/A116/A118/A128.
- The 4,994 maintenance rows form six contiguous episodes lasting 371–1,391 seconds. The shortest gap between episodes is 3,765 seconds. Only one episode has alarm rows in the preceding 30 minutes; this is not enough to establish an alarm predictor.
- Excluding current maintenance and a provisional 30-minute recovery interval after each episode leaves 243,406 pilot rows. The recovery rule is not finalized.
- A chronological 70/15/15 split places 4/2/0 episodes in train/validation/test. A test set without events cannot support meaningful predictive accuracy. Expand the observation period before Naive Bayes or other baseline evaluation.
- Select features using training data only; exclude `m_id`, `m_subid`, and information not available at prediction time. Report accuracy alongside precision, recall, F1, PR-AUC, and event-level warning coverage.
- Candidate prediction horizons remain 30 minutes, 1 hour, and 4 hours. Finalize the target and recovery rule during the later target-design stage, using a broader chronological sample.